# ANALYSE-07 — Percolation critique et Ising sur l'arbre (companion natif)

Sous-série **ANALYSE** (gradation #17545) — Pli 3 de la digestion **Origami** (#19898) : deux résultats de 2026 lus depuis **notre** lac `percolation_lean` — le noyau Harris–Kleitman et l'événement de connexion.

Ce carnet est un companion **natif** : kernel Lean 4 sous WSL, chaque `#check` interroge le lac réellement compilé, chaque `#eval` calcule réellement. Le calcul numérique se fait **en Lean natif** — pas de repli Python, le suffixe du nom suit la convention de la série.

**Prérequis** : le tutoriel de la série principale (carnets 1 à 6). ANALYSE-04 (primitives de PFR) donne le goût des arguments d'inégalité sur un treillis booléen ; il n'est pas exigé.

**Plan** : le socle nôtre (Harris–Kleitman, inégalités FKG) — les deux familles 2026 citées avec leur statut — notre calcul de percolation sur l'arbre $d$-régulier (Galton–Watson, exact) — la distinction de deux seuils que l'on confond souvent — les objets finis du lac — trois exercices.

## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [ANALYSE-04 — Trois primitives de PFR](ANALYSE-04-PFR-Primitives-Python.ipynb) | [ANALYSE-07 — version anglaise](ANALYSE-07-Percolation-Critique_en.ipynb) |

## 0. Statut épistémique — la règle de ce carnet

La digestion Origami exige un **statut déclaré** pour chaque énoncé. Trois statuts existent : **nôtre** (prouvé ou calculé dans notre lac), **formalisée** (prouvée par les auteurs d'openai/math, squelette Lean publié), **non formalisée** (aucune preuve formelle). La règle : jamais « OpenAI a prouvé X, donc X » — ce carnet distingue à chaque ligne ce que **nous prouvons et exécutons** de ce que **nous citons**.

| Objet | Statut | Où le vérifier |
|---|---|---|
| `harris_kleitman_upper_upper`, `harris_kleitman_lower_lower` (inégalités FKG, lac `percolation_lean`) | **nôtre** | `#check` + `#print axioms`, section 1 |
| Famille **213** — criticité Benjamini–Schramm | **formalisée** (par les auteurs) | citée section 2 — squelette `lean/docs/213.md` d'openai/math |
| Famille **236** — seuil factor-of-IID de l'Ising sur l'arbre $d$-régulier | **formalisée** (par les auteurs) | citée section 4 — squelette `lean/docs/236.md` |
| Probabilité critique $p_c = 1/(d-1)$ sur l'arbre $d$-régulier ; seuil de reconstruction $\tanh\beta_c = (d-1)^{-1/2}$ | **calcul nôtre** (Galton–Watson, élémentaire) | sections 3 et 4 — `#eval` natif |

Ce que ce carnet **ne fait pas** : re-prouver les résultats 2026. Leur échelle met la re-preuve hors de portée de ce dépôt — c'est précisément pourquoi le statut « formalisée » existe.

**Un piège de vocabulaire, désamorcé d'avance.** Deux seuils circulent pour l'Ising sur l'arbre $d$-régulier et **ce ne sont pas les mêmes quantités** : le seuil de **reconstruction** (Kesten–Stigum, ce que la famille 236 énonce), $\tanh\beta_c = (d-1)^{-1/2}$ ; et le seuil d'**aimantation** spontanée, $\tanh\beta_c = 1/(d-1)$. La section 4 calcule les deux et montre l'écart. Confondre les deux est l'erreur que ce carnet existe pour éviter.

## 1. Le socle : les inégalités de Harris–Kleitman, prouvées chez nous

Le lac `percolation_lean` porte le noyau combinatoire de la théorie de la percolation : les **inégalités FKG** (Fortuin–Kasteleyn–Ginibre) dans leur cas le plus élémentaire, celui des familles croissantes et décroissantes d'un treillis booléen.

L'énoncé : si $\mathcal{A}$ est une famille **croissante** (ajouter un élément ne fait que conserver l'appartenance) et $\mathcal{B}$ une famille **décroissante**, alors
$$|\mathcal{A}| \cdot |\mathcal{B}| \;\le\; 2^{|\alpha|} \cdot |\mathcal{A} \cap \mathcal{B}|.$$
C'est l'inégalité de Harris–Kleitman, et c'est exactement l'outil qui rend la percolation monotone : l'événement « $u$ est connecté à $v$ » est croissant, l'événement « $\mathcal{A}$ est fermé » est décroissant, et leur corrélation positive est ce qui donne un sens à « il existe un seuil ».

Importons le lac. Ses globs couvrent les **sous-modules** (pas la racine) : nous importons donc explicitement les modules utilisés — l'agrégateur racine `Percolation.lean` n'est pas un module compilé du lac :

In [1]:
import Percolation.Basic
import Percolation.Boundary
import Percolation.Components
import Percolation.Connectivity
import Percolation.Examples

open Percolation

#check Percolation.harris_kleitman_upper_upper
#check Percolation.harris_kleitman_lower_lower
#check Percolation.harris_kleitman_connected

import Percolation.Basic
import Percolation.Boundary
import Percolation.Components
import Percolation.Connectivity
import Percolation.Examples

open Percolation

#check Percolation.harris_kleitman_upper_upper
──────▶  Percolation.harris_kleitman_upper_upper.{u_1} {α : Type u_1} [Fintype α] [DecidableEq α] (𝒜 ℬ : Finset (Finset α))
  (h𝒜 : IsUpperSet ↑𝒜) (hℬ : IsUpperSet ↑ℬ) : 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card
#check Percolation.harris_kleitman_lower_lower
──────▶  Percolation.harris_kleitman_lower_lower.{u_1} {α : Type u_1} [Fintype α] [DecidableEq α] (𝒜 ℬ : Finset (Finset α))
  (h𝒜 : IsLowerSet ↑𝒜) (hℬ : IsLowerSet ↑ℬ) : 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card
#check Percolation.harris_kleitman_connected
──────▶  Percolation.harris_kleitman_connected.{u_1} {V : Type u_1} [Fintype V] [DecidableEq V] (G : SimpleGraph V)
  [Fintype (Edge G)] [DecidableEq (Edge G)] {u v x y : V} :
  (connectedEvent G u v).card * (connectedEvent G x y).card ≤
    2 ^ Fintype.card (Edge G) * (connectedEvent G u v ∩ connectedEvent G x y).card
--% env 0

Raw input:
{"cmd": "import Percolation.Basic\nimport Percolation.Boundary\nimport Percolation.Components\nimport Percolation.Connectivity\nimport Percolation.Examples\n\nopen Percolation\n\n#check Percolation.harris_kleitman_upper_upper\n#check Percolation.harris_kleitman_lower_lower\n#check Percolation.harris_kleitman_connected"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "Percolation.harris_kleitman_upper_upper.{u_1} {α : Type u_1} [Fintype α] [DecidableEq α] (𝒜 ℬ : Finset (Finset α))\n  (h𝒜 : IsUpperSet ↑𝒜) (hℬ : IsUpperSet ↑ℬ) : 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "Percolation.harris_kleitman_lower_lower.{u_1} {α : Type u_1} [Fintype α] [DecidableEq α] (𝒜 ℬ : Finset (Finset α))\n  (h𝒜 : IsLowerSet ↑𝒜) (hℬ : IsLowerSet ↑ℬ) : 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "Percolation.harris_kleitman_connected.{u_1} {V : Type u_1} [Fintype V] [DecidableEq V] (G : SimpleGraph V)\n  [Fintype (Edge G)] [DecidableEq (Edge G)] {u v x y : V} :\n  (connectedEvent G u v).card * (connectedEvent G x y).card ≤\n    2 ^ Fintype.card (Edge G) * (connectedEvent G u v ∩ connectedEvent G x y).card"}],
 "env": 0}

### Lecture du résultat

Les deux premières déclarations sont le socle : `harris_kleitman_upper_upper` pour deux familles croissantes, `harris_kleitman_lower_lower` pour deux familles décroissantes. La troisième, `harris_kleitman_connected`, est la version **percolation** du même noyau — elle porte sur l'événement de connexion, et c'est celle qui parle le langage du carnet. Vérifions leur empreinte logique :

In [2]:
#print axioms Percolation.harris_kleitman_upper_upper
#print axioms Percolation.harris_kleitman_lower_lower
#print axioms Percolation.harris_kleitman_connected

#print axioms Percolation.harris_kleitman_upper_upper
──────▶  'Percolation.harris_kleitman_upper_upper' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms Percolation.harris_kleitman_lower_lower
──────▶  'Percolation.harris_kleitman_lower_lower' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms Percolation.harris_kleitman_connected
──────▶  'Percolation.harris_kleitman_connected' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 1

Raw input:
{"cmd": "#print axioms Percolation.harris_kleitman_upper_upper\n#print axioms Percolation.harris_kleitman_lower_lower\n#print axioms Percolation.harris_kleitman_connected", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'Percolation.harris_kleitman_upper_upper' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'Percolation.harris_kleitman_lower_lower' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'Percolation.harris_kleitman_connected' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 1}

### Sans `sorry`, empreinte minimale

`[propext, Classical.choice, Quot.sound]` — les trois axiomes standards de Mathlib, aucun `sorryAx`, aucun `native_decide`. Le noyau est **nôtre** au sens plein : prouvé dans le lac, vérifiable par cette commande à tout instant.

## 2. Famille 213 — criticité Benjamini–Schramm

**Statut : formalisée** (par les auteurs d'openai/math, 2026). Énoncé cité, jamais exécuté ici.

La conjecture de Benjamini–Schramm prédit l'**absence de percolation à la criticité** : pour la percolation par arêtes sur un graphe infini connexe, localement fini et transitif, s'il existe une probabilité $p$ pour laquelle le cluster ouvert est infini presque sûrement, alors il en va de même à la probabilité critique $p_c$. Autrement dit : le régime critique n'est pas un régime de transition « mou », il est déjà dans la phase connectée dès qu'on atteint $p_c$.

Ce que cela change pour nous : la question « que se passe-t-il *exactement* à $p_c$ ? » n'est pas un détail technique, c'est la question qui a occupé la théorie de la percolation pendant quarante ans. Le résultat 2026 cité ici la résout pour cette classe de graphes.

**Ce que nous ne faisons pas** : ni l'énoncer comme nôtre, ni en donner une preuve. Le squelette `lean/docs/213.md` du dépôt `openai/math` est la référence ; il n'est pas présent dans ce dépôt (vérifié), et nous ne prétendons donc pas l'avoir lu — nous citons l'énoncé tel que la distillation Origami le rapporte.

## 3. Notre calcul — la percolation sur l'arbre $d$-régulier

Voici la partie **nôtre**, et elle est exacte, pas approchée.

Sur l'arbre $d$-régulier, on explore la composante ouverte du sommet racine par un processus de **Galton–Watson** : chaque sommet atteint a $d-1$ enfants (le parent est déjà compté), et chaque arête enfant est ouverte indépendamment avec probabilité $p$. Le nombre moyen d'enfants ouverts vaut donc $(d-1)p$.

Le critère de la progéniture moyenne donne le seuil : la composante est finie si $(d-1)p < 1$, et infinie (avec probabilité positive) si $(d-1)p > 1$. D'où
$$p_c = \frac{1}{d-1}.$$

La probabilité d'extinction obéit à sa propre récursion — la fonction génératrice de la loi binomiale $\mathrm{Bin}(d-1, p)$ au point $q$ :
$$q_{n+1} = \big((1-p) + p\,q_n\big)^{d-1}, \qquad q_0 = 0.$$
Le point fixe non trivial de cette récursion est la probabilité que **l'infection meure avant de s'être propagée**. Écrivons-la en Lean natif :

In [3]:
/-- Puissance entiere sur `Float` : `HPow Float Nat` n'existe pas dans cette
version de Lean (mesure : `failed to synthesize HPow Float ℕ`), et un echec
d'instance rendrait l'expression dependante de l'axiome `sorry` -- `#eval`
abandonnerait alors sans calculer. On l'ecrit donc a la main. -/
def fpow (x : Float) : Nat → Float
  | 0 => 1.0
  | n + 1 => x * fpow x n

/-- Probabilite d'extinction apres `n` generations, pour `b = d - 1` enfants
par sommet. Recursion de Galton-Watson de la loi Bin(b, p). -/
def extinction (p : Float) (b : Nat) : Nat → Float
  | 0 => 0.0
  | k + 1 => fpow ((1.0 - p) + p * extinction p b k) b

-- b = 2  (d = 3) : p_c = 1/2
#eval extinction 0.30 2 80    -- sous-critique -> extinction certaine
#eval extinction 0.50 2 80    -- a p_c exactement (progeniture moyenne = 1) -> tend vers 1
#eval extinction 0.60 2 80    -- sur-critique -> survie de probabilite positive, donc q* < 1

-- b = 3  (d = 4) : p_c = 1/3
#eval extinction 0.25 3 80
#eval extinction 0.40 3 80

-- b = 4  (d = 5) : p_c = 1/4
#eval extinction 0.20 4 80
#eval extinction 0.30 4 80

/-- Puissance entiere sur `Float` : `HPow Float Nat` n'existe pas dans cette
version de Lean (mesure : `failed to synthesize HPow Float ℕ`), et un echec
d'instance rendrait l'expression dependante de l'axiome `sorry` -- `#eval`
abandonnerait alors sans calculer. On l'ecrit donc a la main. -/
def fpow (x : Float) : Nat → Float
  | 0 => 1.0
  | n + 1 => x * fpow x n

/-- Probabilite d'extinction apres `n` generations, pour `b = d - 1` enfants
par sommet. Recursion de Galton-Watson de la loi Bin(b, p). -/
def extinction (p : Float) (b : Nat) : Nat → Float
  | 0 => 0.0
  | k + 1 => fpow ((1.0 - p) + p * extinction p b k) b

-- b = 2  (d = 3) : p_c = 1/2
#eval extinction 0.30 2 80    -- sous-critique -> extinction certaine
─────▶  1.000000
#eval extinction 0.50 2 80    -- a p_c exactement (progeniture moyenne = 1) -> tend vers 1
─────▶  0.954145
#eval extinction 0.60 2 80    -- sur-critique -> survie de probabilite positive, donc q* < 1
─────▶  0.444444

-- b = 3  (d = 4) : p_c = 1/3
#eval extinction 0.25 3 80
─────▶  1.000000
#eval extinction 0.40 3 80
─────▶  0.557189

-- b = 4  (d = 5) : p_c = 1/4
#eval extinction 0.20 4 80
─────▶  1.000000
#eval extinction 0.30 4 80
─────▶  0.598335
--% env 2

Raw input:
{"cmd": "/-- Puissance entiere sur `Float` : `HPow Float Nat` n'existe pas dans cette\nversion de Lean (mesure : `failed to synthesize HPow Float \u2115`), et un echec\nd'instance rendrait l'expression dependante de l'axiome `sorry` -- `#eval`\nabandonnerait alors sans calculer. On l'ecrit donc a la main. -/\ndef fpow (x : Float) : Nat \u2192 Float\n  | 0 => 1.0\n  | n + 1 => x * fpow x n\n\n/-- Probabilite d'extinction apres `n` generations, pour `b = d - 1` enfants\npar sommet. Recursion de Galton-Watson de la loi Bin(b, p). -/\ndef extinction (p : Float) (b : Nat) : Nat \u2192 Float\n  | 0 => 0.0\n  | k + 1 => fpow ((1.0 - p) + p * extinction p b k) b\n\n-- b = 2  (d = 3) : p_c = 1/2\n#eval extinction 0.30 2 80    -- sous-critique -> extinction certaine\n#eval extinction 0.50 2 80    -- a p_c exactement (progeniture moyenne = 1) -> tend vers 1\n#eval extinction 0.60 2 80    -- sur-critique -> survie de probabilite positive, donc q* < 1\n\n-- b = 3  (d = 4) : p_c = 1/3\n#eval extinction 0.25 3 80\n#eval extinction 0.40 3 80\n\n-- b = 4  (d = 5) : p_c = 1/4\n#eval extinction 0.20 4 80\n#eval extinction 0.30 4 80", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "1.000000"},
  {"severity": "info",
   "pos": {"line": 17, "column": 0},
   "endPos": {"line": 17, "column": 5},
   "data": "0.954145"},
  {"severity": "info",
   "pos": {"line": 18, "column": 0},
   "endPos": {"line": 18, "column": 5},
   "data": "0.444444"},
  {"severity": "info",
   "pos": {"line": 21, "column": 0},
   "endPos": {"line": 21, "column": 5},
   "data": "1.000000"},
  {"severity": "info",
   "pos": {"line": 22, "column": 0},
   "endPos": {"line": 22, "column": 5},
   "data": "0.557189"},
  {"severity": "info",
   "pos": {"line": 25, "column": 0},
   "endPos": {"line": 25, "column": 5},
   "data": "1.000000"},
  {"severity": "info",
   "pos": {"line": 26, "column": 0},
   "endPos": {"line": 26, "column": 5},
   "data": "0.598335"}],
 "env": 2}

### Lecture du résultat

Trois lectures, du plus simple au plus intéressant.

**Sous-critique, l'extinction est certaine.** En dessous de $p_c$, les trois premières valeurs valent $1.000000$ : la composante ouverte est finie presque sûrement. C'est le régime où le cluster ne « percole » pas.

**À $p_c$ exactement, la valeur est subtile.** Pour $b=2$ et $p=0.5$, la progéniture moyenne vaut exactement $1$ : c'est le cas critique, et le calcul rend $0.954145$ après 80 itérations — proche de $1$ sans l'atteindre, et **convergeant lentement**. Cette lenteur n'est pas un artefact : à la criticité, la décroissance de la probabilité de survie est polynomiale, pas exponentielle. C'est précisément le régime dont la famille 213 parle.

**Sur-critique, le point fixe est exactement $4/9$.** Pour $b=2$, $p=0.6$, le calcul rend $0.444444$. Résolvons à la main pour vérifier : $q = (0.4 + 0.6q)^2$ donne $0.36q^2 - 0.52q + 0.16 = 0$, soit $9q^2 - 13q + 4 = 0$, dont les racines sont $q = 1$ et $q = 4/9$. Le point fixe non trivial est donc **exactement $4/9$** — et c'est la valeur que le compilateur produit, sur les définitions mêmes du carnet. Une valeur sortie du calcul et vérifiable à la main : c'est ce qu'on demande à un `#eval` pédagogique.

## 4. Famille 236 — le seuil de l'Ising sur l'arbre, et les deux quantités qu'on confond

**Statut : formalisée** (par les auteurs d'openai/math, 2026). Énoncé cité, jamais exécuté ici.

La famille 236 donne le **seuil exact factor-of-IID** pour l'Ising libre sur l'arbre $d$-régulier : la propriété tient **exactement** lorsque
$$\tanh\beta \;\le\; (d-1)^{-1/2},$$
égalité incluse, pour $d \ge 3$ et $\beta \ge 0$ — la question, posée par Lyons, dans le cas ferromagnétique.

**Le piège, et il est classique.** Un autre seuil circule pour le même objet : $\tanh\beta_c = 1/(d-1)$. Les deux sont réels, les deux sont des seuils de l'Ising sur l'arbre $d$-régulier, et **ils ne mesurent pas la même chose** :

| Seuil | Ce qu'il caractérise | Origine |
|---|---|---|
| $\tanh\beta_c = (d-1)^{-1/2}$ | **reconstruction** : l'état de la racine reste-t-il corrélé à celui de la frontière lointaine ? | Kesten–Stigum ; c'est l'objet de la famille 236 |
| $\tanh\beta_c = 1/(d-1)$ | **aimantation** spontanée : la phase ordonnée apparaît-elle ? | critère de Bethe / champ moyen sur l'arbre |

Concrètement, $(d-1)^{-1/2} > 1/(d-1)$ pour $d \ge 3$ : le seuil de reconstruction est **plus haut**. Entre les deux, il existe une phase où l'aimantation est nulle mais où l'information de la frontière est pourtant reconstructible. C'est ce régime intermédiaire qui rend la question de Lyons intéressante — et c'est pourquoi présenter les deux sous un seul nom serait une faute, pas une simplification.

**Ce que nous ne faisons pas** : dériver le seuil cité. Ce qui suit est un **calcul élémentaire nôtre** — l'algèbre des deux seuils et leur écart sur petits $d$, exécuté par le compilateur. C'est une vérification de cohérence, **pas** une re-preuve de la famille 236.

In [4]:
-- Deux pieges de coercion, tous deux MESURES au preflight -- et tous deux du
-- meme genre : l'echec d'instance ne dit pas « erreur de type », il rend la
-- definition dependante de l'axiome `sorry`, et les `#eval` en aval AVORTENT
-- au lieu de calculer. Un faux « echec de calcul » qui est une erreur de type.
--
--   (1) `HPow Float Nat` n'existe pas : d'ou `fpow` ecrit a la main ci-dessus.
--   (2) `(d - 1 : Float)` sur `d : Nat` ne se coerced PAS (Lean cherche
--       `HSub Nat Nat Float`) ; `((d - 1 : Nat) : Float)` passe l'analyse mais
--       laisse ensuite un `Nat` la ou un `Float` est attendu.
--
-- La forme qui marche est explicite : `Float.ofNat (d - 1)` -- soustraction
-- dans Nat, puis conversion nommee. Aucune inference implicite en jeu.

/-- Seuil de reconstruction (Kesten-Stigum) : `tanh beta = (d-1)^(-1/2)`.
Egal a la racine positive de `(d-1) * t^2 = 1`. -/
def seuilReconstruction (d : Nat) : Float :=
  1.0 / Float.sqrt (Float.ofNat (d - 1))

/-- Seuil d'aimantation (critere de Bethe) : `tanh beta = 1/(d-1)`.
A ne PAS confondre avec le precedent : c'est l'objet de la colonne de droite
du tableau de la section 4. -/
def seuilAimantation (d : Nat) : Float :=
  1.0 / Float.ofNat (d - 1)

/-- Verifie l'identite algebrique `(d-1) * t^2 = 1` pour `t` le seuil de
reconstruction : c'est la forme sous laquelle Kesten-Stigum l'ecrit. -/
def verifieKS (d : Nat) : Float :=
  Float.ofNat (d - 1) * seuilReconstruction d * seuilReconstruction d

-- Les deux seuils, cote a cote, sur petits d (d >= 3 -> b = d-1 >= 2)
#eval (seuilReconstruction 3, seuilAimantation 3)   -- (0.707107, 0.500000)
#eval (seuilReconstruction 4, seuilAimantation 4)   -- (0.577350, 0.333333)
#eval (seuilReconstruction 5, seuilAimantation 5)   -- (0.500000, 0.250000)
#eval (seuilReconstruction 6, seuilAimantation 6)

-- L'identite de Kesten-Stigum, verifiee numeriquement : doit rendre 1.0
#eval verifieKS 3
#eval verifieKS 4
#eval verifieKS 5

-- Le seuil de reconstruction est STRICTEMENT au-dessus de celui d'aimantation
#eval (seuilReconstruction 3 > seuilAimantation 3)
#eval (seuilReconstruction 10 > seuilAimantation 10)

-- Deux pieges de coercion, tous deux MESURES au preflight -- et tous deux du
-- meme genre : l'echec d'instance ne dit pas « erreur de type », il rend la
-- definition dependante de l'axiome `sorry`, et les `#eval` en aval AVORTENT
-- au lieu de calculer. Un faux « echec de calcul » qui est une erreur de type.
--
--   (1) `HPow Float Nat` n'existe pas : d'ou `fpow` ecrit a la main ci-dessus.
--   (2) `(d - 1 : Float)` sur `d : Nat` ne se coerced PAS (Lean cherche
--       `HSub Nat Nat Float`) ; `((d - 1 : Nat) : Float)` passe l'analyse mais
--       laisse ensuite un `Nat` la ou un `Float` est attendu.
--
-- La forme qui marche est explicite : `Float.ofNat (d - 1)` -- soustraction
-- dans Nat, puis conversion nommee. Aucune inference implicite en jeu.

/-- Seuil de reconstruction (Kesten-Stigum) : `tanh beta = (d-1)^(-1/2)`.
Egal a la racine positive de `(d-1) * t^2 = 1`. -/
def seuilReconstruction (d : Nat) : Float :=
  1.0 / Float.sqrt (Float.ofNat (d - 1))

/-- Seuil d'aimantation (critere de Bethe) : `tanh beta = 1/(d-1)`.
A ne PAS confondre avec le precedent : c'est l'objet de la colonne de droite
du tableau de la section 4. -/
def seuilAimantation (d : Nat) : Float :=
  1.0 / Float.ofNat (d - 1)

/-- Verifie l'identite algebrique `(d-1) * t^2 = 1` pour `t` le seuil de
reconstruction : c'est la forme sous laquelle Kesten-Stigum l'ecrit. -/
def verifieKS (d : Nat) : Float :=
  Float.ofNat (d - 1) * seuilReconstruction d * seuilReconstruction d

-- Les deux seuils, cote a cote, sur petits d (d >= 3 -> b = d-1 >= 2)
#eval (seuilReconstruction 3, seuilAimantation 3)   -- (0.707107, 0.500000)
─────▶  (0.707107, 0.500000)
#eval (seuilReconstruction 4, seuilAimantation 4)   -- (0.577350, 0.333333)
─────▶  (0.577350, 0.333333)
#eval (seuilReconstruction 5, seuilAimantation 5)   -- (0.500000, 0.250000)
─────▶  (0.500000, 0.250000)
#eval (seuilReconstruction 6, seuilAimantation 6)
─────▶  (0.447214, 0.200000)

-- L'identite de Kesten-Stigum, verifiee numeriquement : doit rendre 1.0
#eval verifieKS 3
─────▶  1.000000
#eval verifieKS 4
─────▶  1.000000
#eval verifieKS 5
─────▶  1.000000

-- Le seuil de reconstruction est STRICTEMENT au-dessus de celui d'aimantation
#eval (seuilReconstruction 3 > seuilAimantation 3)
─────▶  true
#eval (seuilReconstruction 10 > seuilAimantation 10)
─────▶  true
--% env 3

Raw input:
{"cmd": "-- Deux pieges de coercion, tous deux MESURES au preflight -- et tous deux du\n-- meme genre : l'echec d'instance ne dit pas \u00ab erreur de type \u00bb, il rend la\n-- definition dependante de l'axiome `sorry`, et les `#eval` en aval AVORTENT\n-- au lieu de calculer. Un faux \u00ab echec de calcul \u00bb qui est une erreur de type.\n--\n--   (1) `HPow Float Nat` n'existe pas : d'ou `fpow` ecrit a la main ci-dessus.\n--   (2) `(d - 1 : Float)` sur `d : Nat` ne se coerced PAS (Lean cherche\n--       `HSub Nat Nat Float`) ; `((d - 1 : Nat) : Float)` passe l'analyse mais\n--       laisse ensuite un `Nat` la ou un `Float` est attendu.\n--\n-- La forme qui marche est explicite : `Float.ofNat (d - 1)` -- soustraction\n-- dans Nat, puis conversion nommee. Aucune inference implicite en jeu.\n\n/-- Seuil de reconstruction (Kesten-Stigum) : `tanh beta = (d-1)^(-1/2)`.\nEgal a la racine positive de `(d-1) * t^2 = 1`. -/\ndef seuilReconstruction (d : Nat) : Float :=\n  1.0 / Float.sqrt (Float.ofNat (d - 1))\n\n/-- Seuil d'aimantation (critere de Bethe) : `tanh beta = 1/(d-1)`.\nA ne PAS confondre avec le precedent : c'est l'objet de la colonne de droite\ndu tableau de la section 4. -/\ndef seuilAimantation (d : Nat) : Float :=\n  1.0 / Float.ofNat (d - 1)\n\n/-- Verifie l'identite algebrique `(d-1) * t^2 = 1` pour `t` le seuil de\nreconstruction : c'est la forme sous laquelle Kesten-Stigum l'ecrit. -/\ndef verifieKS (d : Nat) : Float :=\n  Float.ofNat (d - 1) * seuilReconstruction d * seuilReconstruction d\n\n-- Les deux seuils, cote a cote, sur petits d (d >= 3 -> b = d-1 >= 2)\n#eval (seuilReconstruction 3, seuilAimantati

### Lecture du résultat

Les trois couples confirment le tableau, et la dernière paire de valeurs dit pourquoi le piège mérite d'être désamorcé explicitement :

- $d = 3$ : reconstruction $0.707107$, aimantation $0.500000$ ;
- $d = 4$ : $0.577350$ contre $0.333333$ ;
- $d = 5$ : $0.500000$ contre $0.250000$.

Le seuil de reconstruction est **au-dessus** du seuil d'aimantation pour tout $d \ge 3$, et l'écart se creuse avec $d$ : $1/\sqrt{2}$ contre $1/2$, puis $1/\sqrt{3}$ contre $1/3$… Les deux quantités sont donc non seulement distinctes, mais **séparées par un intervalle non vide** — c'est là que vit le régime « aimantation nulle, reconstruction possible ».

L'identité de Kesten–Stigum $(d-1)\,t^2 = 1$ rend bien $1.000000$ dans les trois cas : le seuil calculé est exactement la racine positive de cette équation, ce qui est la forme sous laquelle la littérature l'écrit.

**Rappel du statut** : ceci est **notre** calcul élémentaire. La famille 236 énonce le résultat *exact* (inégalité, égalité incluse) et le prouve ; nous vérifions que l'algèbre du seuil cité est cohérente et qu'elle n'est pas celle de l'aimantation. Ce n'est pas la même chose, et le carnet ne prétend pas le contraire.

## 5. Les objets finis du lac — voir la théorie sur $C_3$ et $C_4$

Avant de conclure, redescendons du graphe infini aux objets finis que le lac expose : le cycle $C_3$ et le cycle $C_4$. Ils ne percolent pas (un cycle fini n'a pas de composante infinie), mais ils portent les définitions exactes — bord ouvert, ensemble fermé, connexité — et permettent de **voir** les énoncés sur des cas qu'on peut énumérer.

In [5]:
#check Percolation.card_edge_C3
#check Percolation.C3_closed_iff
#check Percolation.boundary_card_C4_opposite
#check Percolation.boundary_attains_min_C4

#check Percolation.card_edge_C3
──────▶  Percolation.card_edge_C3 : Fintype.card (Edge C3) = 3
#check Percolation.C3_closed_iff
──────▶  Percolation.C3_closed_iff (A : Set (Fin 3)) : openEdgeClosed C3 full3 A ↔ A = Set.univ ∨ A = ∅
#check Percolation.boundary_card_C4_opposite
──────▶  Percolation.boundary_card_C4_opposite : (boundary C4 full4 {0, 2}).card = 4
#check Percolation.boundary_attains_min_C4
──────▶  Percolation.boundary_attains_min_C4 : ∃ A, A.Nonempty ∧ A ≠ Finset.univ ∧ (boundary C4 full4 A).card = 2
--% env 4

Raw input:
{"cmd": "#check Percolation.card_edge_C3\n#check Percolation.C3_closed_iff\n#check Percolation.boundary_card_C4_opposite\n#check Percolation.boundary_attains_min_C4", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Percolation.card_edge_C3 : Fintype.card (Edge C3) = 3"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Percolation.C3_closed_iff (A : Set (Fin 3)) : openEdgeClosed C3 full3 A ↔ A = Set.univ ∨ A = ∅"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Percolation.boundary_card_C4_opposite : (boundary C4 full4 {0, 2}).card = 4"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Percolation.boundary_attains_min_C4 : ∃ A, A.Nonempty ∧ A ≠ Finset.univ ∧ (boundary C4 full4 A).card = 2"}],
 "env": 4}

### Lecture du résultat

`card_edge_C3` compte les arêtes du triangle : $3$, comme attendu. `C3_closed_iff` est le résultat le plus parlant du lot — il caractérise **exactement** les ensembles fermés du triangle, c'est-à-dire ceux qu'aucune arête ouverte ne traverse. Sur un graphe de trois sommets, cette caractérisation s'énonce et se vérifie complètement : c'est le plus petit laboratoire où l'on peut regarder un « bord » sans tricher.

`boundary_card_C4_opposite` est le contre-exemple pédagogique du carnet : dans $C_4$, le bord de deux sommets **opposés** a $4$ arêtes, alors que celui de deux sommets **adjacents** n'en a que $2$. Le bord n'est donc pas une fonction de la taille de l'ensemble — la géométrie compte. C'est ce genre d'asymétrie qui, sur l'arbre infini, devient la différence entre percoler et ne pas percoler.

`boundary_attains_min_C4` ferme l'arc : sur $C_4$, le minimum du bord est **atteint**, et il est atteint par un ensemble précis. Un énoncé d'existence, vérifié par le lac.

## Exercices

### Exercice 1 — Retrouver $p_c$ sans le calcul (sur papier)

Nous avons obtenu $p_c = 1/(d-1)$ par le critère de la progéniture moyenne. Retrouvez-le **directement** sur la récursion $q_{n+1} = ((1-p) + p\,q_n)^{d-1}$ : montrez que la tangente en $q = 0$ a pour pente $(d-1)p$, et que le point fixe $q = 1$ cesse d'être attractif exactement lorsque cette pente dépasse $1$. Vérifiez sur $b = 2$ : à $p = 0.5$ la pente vaut $1$ (cas critique), à $p = 0.6$ elle vaut $1.2$ (sur-critique). Puis confrontez votre prédiction aux `#eval` de la section 3 — l'accord est-il exact ?

In [6]:
-- Exercice 1 -- retrouver p_c = 1/(d-1) par la pente en q = 0
--
-- Etape 1 : la pente en q = 0 de q |-> ((1-p) + p*q)^b vaut b*p ; justifier.
-- Etape 2 : q = 1 cesse d'etre attractif des que b*p > 1 ; en deduire
--            p_c = 1/b, puis p_c = 1/(d-1) avec b = d - 1.
-- Etape 3 : confronter la prediction aux `#eval` de la section 3 (b = 2 :
--            p = 0.5 rend 0.954145, p = 0.6 rend 0.444444). L'accord ?
--
-- TODO etudiant : remplacer le corps de `penteEnZero`, puis re-executer.

def penteEnZero (p : Float) (b : Nat) : Float :=
  Float.ofNat b * p * 0.0  -- TODO etudiant : retirer le facteur `* 0.0`

#eval penteEnZero 0.30 2    -- une fois rempli : 0.600000 (sous-critique)
#eval penteEnZero 0.50 2    -- une fois rempli : 1.000000 (cas critique)
#eval penteEnZero 0.60 2    -- une fois rempli : 1.200000 (sur-critique)

-- Exercice 1 -- retrouver p_c = 1/(d-1) par la pente en q = 0
--
-- Etape 1 : la pente en q = 0 de q |-> ((1-p) + p*q)^b vaut b*p ; justifier.
-- Etape 2 : q = 1 cesse d'etre attractif des que b*p > 1 ; en deduire
--            p_c = 1/b, puis p_c = 1/(d-1) avec b = d - 1.
-- Etape 3 : confronter la prediction aux `#eval` de la section 3 (b = 2 :
--            p = 0.5 rend 0.954145, p = 0.6 rend 0.444444). L'accord ?
--
-- TODO etudiant : remplacer le corps de `penteEnZero`, puis re-executer.

def penteEnZero (p : Float) (b : Nat) : Float :=
  Float.ofNat b * p * 0.0  -- TODO etudiant : retirer le facteur `* 0.0`

#eval penteEnZero 0.30 2    -- une fois rempli : 0.600000 (sous-critique)
─────▶  0.000000
#eval penteEnZero 0.50 2    -- une fois rempli : 1.000000 (cas critique)
─────▶  0.000000
#eval penteEnZero 0.60 2    -- une fois rempli : 1.200000 (sur-critique)
─────▶  0.000000
--% env 5

Raw input:
{"cmd": "-- Exercice 1 -- retrouver p_c = 1/(d-1) par la pente en q = 0\n--\n-- Etape 1 : la pente en q = 0 de q |-> ((1-p) + p*q)^b vaut b*p ; justifier.\n-- Etape 2 : q = 1 cesse d'etre attractif des que b*p > 1 ; en deduire\n--            p_c = 1/b, puis p_c = 1/(d-1) avec b = d - 1.\n-- Etape 3 : confronter la prediction aux `#eval` de la section 3 (b = 2 :\n--            p = 0.5 rend 0.954145, p = 0.6 rend 0.444444). L'accord ?\n--\n-- TODO etudiant : remplacer le corps de `penteEnZero`, puis re-executer.\n\ndef penteEnZero (p : Float) (b : Nat) : Float :=\n  Float.ofNat b * p * 0.0  -- TODO etudiant : retirer le facteur `* 0.0`\n\n#eval penteEnZero 0.30 2    -- une fois rempli : 0.600000 (sous-critique)\n#eval penteEnZero 0.50 2    -- une fois rempli : 1.000000 (cas critique)\n#eval penteEnZero 0.60 2    -- une fois rempli : 1.200000 (sur-critique)", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "0.000000"}],
 "env": 5}

### Exercice 2 — Le point fixe sur-critique, à la main puis en Lean

Pour $b = 2$ et $p = 0.6$, nous avons annoncé $q^* = 4/9$. (a) Refaites la résolution : $q = (0.4 + 0.6q)^2$, développez, montrez que l'équation se réduit à $9q^2 - 13q + 4 = 0$. (b) Factorisez et identifiez les deux racines. (c) Pourquoi la racine pertinente est-elle $4/9$ et non $1$ ? (d) Ajoutez un `#eval` qui itère $200$ générations au lieu de $80$ : la valeur change-t-elle ? Que vous dit ce résultat sur la convergence vers un point fixe **stable** ?

In [7]:
-- Exercice 2 -- le point fixe sur-critique : 4/9, a la main puis en Lean
--
-- Etape 1 : resoudre q = (0.4 + 0.6*q)^2 ; montrer 9q^2 - 13q + 4 = 0.
-- Etape 2 : factoriser et identifier les deux racines (1 et 4/9).
-- Etape 3 : justifier que la racine pertinente est 4/9, et non 1.
-- Etape 4 : iterer 200 generations au lieu de 80, puis comparer.
--
-- TODO etudiant : remplacer la valeur de `qSurCritique` par le point fixe
-- non trivial, puis confronter a la valeur a 80 generations (`0.444444`).

def qSurCritique : Float :=
  0.0  -- TODO etudiant : q tel que q = (0.4 + 0.6*q)^2, et q < 1

#eval qSurCritique
#check @extinction
-- TODO etudiant : ajouter `#eval extinction 0.60 2 200`

-- Exercice 2 -- le point fixe sur-critique : 4/9, a la main puis en Lean
--
-- Etape 1 : resoudre q = (0.4 + 0.6*q)^2 ; montrer 9q^2 - 13q + 4 = 0.
-- Etape 2 : factoriser et identifier les deux racines (1 et 4/9).
-- Etape 3 : justifier que la racine pertinente est 4/9, et non 1.
-- Etape 4 : iterer 200 generations au lieu de 80, puis comparer.
--
-- TODO etudiant : remplacer la valeur de `qSurCritique` par le point fixe
-- non trivial, puis confronter a la valeur a 80 generations (`0.444444`).

def qSurCritique : Float :=
  0.0  -- TODO etudiant : q tel que q = (0.4 + 0.6*q)^2, et q < 1

#eval qSurCritique
─────▶  0.000000
#check @extinction
──────▶  extinction : Float → ℕ → ℕ → Float
-- TODO etudiant : ajouter `#eval extinction 0.60 2 200`
--% env 6

Raw input:
{"cmd": "-- Exercice 2 -- le point fixe sur-critique : 4/9, a la main puis en Lean\n--\n-- Etape 1 : resoudre q = (0.4 + 0.6*q)^2 ; montrer 9q^2 - 13q + 4 = 0.\n-- Etape 2 : factoriser et identifier les deux racines (1 et 4/9).\n-- Etape 3 : justifier que la racine pertinente est 4/9, et non 1.\n-- Etape 4 : iterer 200 generations au lieu de 80, puis comparer.\n--\n-- TODO etudiant : remplacer la valeur de `qSurCritique` par le point fixe\n-- non trivial, puis confronter a la valeur a 80 generations (`0.444444`).\n\ndef qSurCritique : Float :=\n  0.0  -- TODO etudiant : q tel que q = (0.4 + 0.6*q)^2, et q < 1\n\n#eval qSurCritique\n#check @extinction\n-- TODO etudiant : ajouter `#eval extinction 0.60 2 200`", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 6},
   "data": "extinction : Float → ℕ → ℕ → Float"}],
 "env": 6}

### Exercice 3 — Séparer les deux seuils (sur papier)

La section 4 affirme que le seuil de reconstruction $(d-1)^{-1/2}$ est **strictement supérieur** au seuil d'aimantation $1/(d-1)$ pour tout $d \ge 3$. Démontrez-le sur papier : élève au carré, réduis au même dénominateur, et conclus par un argument de monotonie. Combien vaut le rapport des deux seuils quand $d \to \infty$ ? Interprétez : que devient l'intervalle « aimantation nulle, reconstruction possible » sur un arbre de très grand degré ?

In [8]:
-- Exercice 3 -- separer les deux seuils (d-1)^(-1/2) et 1/(d-1)
--
-- Etape 1 : montrer (d-1)^(-1/2) > 1/(d-1) pour tout d >= 3 -- elever au carre.
-- Etape 2 : rapport des deux seuils quand d tend vers l'infini.
-- Etape 3 : interpreter l'intervalle « aimantation nulle, reconstruction
--            possible » sur un arbre de tres grand degre.
--
-- TODO etudiant : remplacer le corps de `rapportSeuils`, puis re-executer.

def rapportSeuils (d : Nat) : Float :=
  seuilReconstruction d / seuilAimantation d * 0.0  -- TODO : retirer `* 0.0`

#eval rapportSeuils 3      -- une fois rempli : 1.414214 (racine de 2)
#eval rapportSeuils 10     -- une fois rempli : 3.000000
#check @seuilReconstruction
#check @seuilAimantation

-- Exercice 3 -- separer les deux seuils (d-1)^(-1/2) et 1/(d-1)
--
-- Etape 1 : montrer (d-1)^(-1/2) > 1/(d-1) pour tout d >= 3 -- elever au carre.
-- Etape 2 : rapport des deux seuils quand d tend vers l'infini.
-- Etape 3 : interpreter l'intervalle « aimantation nulle, reconstruction
--            possible » sur un arbre de tres grand degre.
--
-- TODO etudiant : remplacer le corps de `rapportSeuils`, puis re-executer.

def rapportSeuils (d : Nat) : Float :=
  seuilReconstruction d / seuilAimantation d * 0.0  -- TODO : retirer `* 0.0`

#eval rapportSeuils 3      -- une fois rempli : 1.414214 (racine de 2)
─────▶  0.000000
#eval rapportSeuils 10     -- une fois rempli : 3.000000
─────▶  0.000000
#check @seuilReconstruction
──────▶  seuilReconstruction : ℕ → Float
#check @seuilAimantation
──────▶  seuilAimantation : ℕ → Float
--% env 7

Raw input:
{"cmd": "-- Exercice 3 -- separer les deux seuils (d-1)^(-1/2) et 1/(d-1)\n--\n-- Etape 1 : montrer (d-1)^(-1/2) > 1/(d-1) pour tout d >= 3 -- elever au carre.\n-- Etape 2 : rapport des deux seuils quand d tend vers l'infini.\n-- Etape 3 : interpreter l'intervalle \u00ab aimantation nulle, reconstruction\n--            possible \u00bb sur un arbre de tres grand degre.\n--\n-- TODO etudiant : remplacer le corps de `rapportSeuils`, puis re-executer.\n\ndef rapportSeuils (d : Nat) : Float :=\n  seuilReconstruction d / seuilAimantation d * 0.0  -- TODO : retirer `* 0.0`\n\n#eval rapportSeuils 3      -- une fois rempli : 1.414214 (racine de 2)\n#eval rapportSeuils 10     -- une fois rempli : 3.000000\n#check @seuilReconstruction\n#check @seuilAimantation", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "0.000000"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 6},
   "data": "seuilReconstruction : ℕ → Float"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 6},
   "data": "seuilAimantation : ℕ → Float"}],
 "env": 7}

## Conclusion

Ce carnet a tenu les trois registres de la digestion Origami :

1. **Le socle nôtre** — les inégalités de Harris–Kleitman, prouvées dans notre lac `percolation_lean`, `#check` et `#print axioms` à l'appui : statut **nôtre**, empreinte `[propext, Classical.choice, Quot.sound]`.
2. **Les familles 2026 citées** — 213 (criticité Benjamini–Schramm) et 236 (seuil factor-of-IID de l'Ising sur l'arbre) : énoncés repris fidèlement, statut **formalisée** (par les auteurs), squelettes `lean/docs/NNN.md` en référence — jamais exécutés ni re-prouvés ici.
3. **Notre contribution** — deux calculs élémentaires mais **exacts**, exécutés par le compilateur : la probabilité critique $p_c = 1/(d-1)$ de la percolation sur l'arbre $d$-régulier, avec son point fixe non trivial vérifiable à la main ($4/9$ pour $d=3$, $p=0.6$) ; et la mise au point sur les **deux seuils distincts** de l'Ising sur l'arbre, avec leur identité de Kesten–Stigum vérifiée numériquement.

**Verdicts SOTA** (règle du dépôt) : **SOTA-OK** pour ce carnet — vrai Lean 4 + Mathlib, vrai lac compilé, vrais `#eval` natifs, aucune substitution. **INTRINSIC** pour la re-preuve des familles 2026 : hors d'échelle de ce dépôt — l'existence des squelettes publiés est précisément ce qui rend la citation le geste honnête.

**Références** : lac [`percolation_lean`](../../../Probas/Applications/Percolation/percolation_lean/) · issue #19898 (Epic Origami) et #19920 (Pli 3) · Harris, *A lower bound for the critical probability in a certain percolation process* (1960) · Fortuin, Kasteleyn & Ginibre (1971) · Kesten & Stigum (1966) · Lyons (question du seuil factor-of-IID) · familles 213 et 236 du dépôt openai/math (2026) et leurs squelettes `lean/docs/NNN.md`.